## 07 - Definition and computation of polarization metrics

We want to compute the two polarization metrics that we have defined:

1. Relative frequency of **polarization associated lexicon**.

2. Percentage of accusations per sentence.


> **INPUTS:**
- CSV file containing interventions, tagged by number of sentences, and number of acussations. 

    > `interventions_filtered_tagged.csv`
- Pickle file with a dictionary pointing to Counter objects (similar to dict), containing the probability distribution for each year.

    > `word_frequencies.pkl`

> **OUTPUTS:**

- CSV file containing all the computed metrics.


In [16]:
import pandas as pd
import pickle
from pathlib import Path
from collections import Counter

In [21]:
# Load input data
DATA_DIR = Path.cwd().parent / "data" / "processed"

# Load interventions with tags
interventions_df = pd.read_csv(DATA_DIR / "interventions_filtered_tagged.csv")

# Load word frequency distributions
with open(DATA_DIR / "word_frequencies.pkl", "rb") as f:
    word_frequencies = pickle.load(f)

print(f"Loaded {len(interventions_df)} interventions")
print(f"Columns: {interventions_df.columns.tolist()}")

Loaded 144209 interventions
Columns: ['text', 'group', 'gender', 'year', 'number_of_accusations', 'number_of_sentences']


In [22]:
# Define polarization-associated lexicon
POLARIZATION_LEXICON  = {"corrupción", "corrupto", "corruptela", "demagogia", "demagogo", "rata", "serpiente", "cucaracha", "traidor", "traicionar", "basura", "mentira", "miente", "mentiroso", "falso", "falsedad", "engaño", "tramposo", "sucio", "repugnante", "ridículo", "ridiculez", "cáncer", "tumor", "extirpar", "memo", "fascista", "fascistas", "fascismo", "dictadura", "dictador"}

def compute_polarization_lexicon_metric(row, word_freq_dict):
    """
    Computes the relative frequency of polarization-associated words in interventions.
    
    Parameters:
    -----------
    row : pd.Series
        Row from interventions dataframe containing 'year' and 'group' columns
    word_freq_dict : dict
        Dictionary mapping year -> group -> Counter(tokens)
    
    Returns:
    --------
    float : Relative frequency of polarization words (0-1)
    """
    year = int(row['year'])
    group = row['group']
    
    # Get frequency distribution for this year and group
    if year not in word_freq_dict or group not in word_freq_dict[year]:
        return 0.0
    
    freq_counter = word_freq_dict[year][group]
    total_words = freq_counter.get('__TOTAL_WORDS__', 1)
    
    # Count polarization-associated words
    polarization_count = sum(
        freq_counter.get(word, 0) for word in POLARIZATION_LEXICON
    )
    
    return polarization_count / total_words if total_words > 0 else 0.0

def compute_accusations_per_sentence_metric(row):
    """
    Computes the percentage of accusations per sentence.
    
    Parameters:
    -----------
    row : pd.Series
        Row containing 'num_sentences' and 'num_accusations' columns
    
    Returns:
    --------
    float : Percentage of accusations per sentence (0-100)
    """
    num_sentences = row.get('num_sentences', 1)
    num_accusations = row.get('num_accusations', 0)
    
    if num_sentences == 0:
        return 0.0
    
    return (num_accusations / num_sentences) * 100

In [23]:
# Apply Metric 1: Polarization Lexicon Frequency
print("Computing polarization lexicon metric...")
interventions_df['polarization_lexicon_freq'] = interventions_df.apply(
    lambda row: compute_polarization_lexicon_metric(row, word_frequencies), 
    axis=1
)

# Apply Metric 2: Accusations per Sentence
print("Computing accusations per sentence metric...")
interventions_df['accusations_per_sentence_pct'] = interventions_df.apply(
    compute_accusations_per_sentence_metric, 
    axis=1
)

print("✓ Metrics computed successfully")

Computing polarization lexicon metric...
Computing accusations per sentence metric...
✓ Metrics computed successfully


In [24]:
# Save results
output_path = DATA_DIR / "polarization_metrics.csv"
interventions_df.to_csv(output_path, index=False, encoding='utf-8')

print(f"✓ Metrics saved to {output_path}")
print(f"\nDataset shape: {interventions_df.shape}")
print(f"\nMetrics summary:")
print(interventions_df[['group', 'year', 'polarization_lexicon_freq', 'accusations_per_sentence_pct']].describe())

✓ Metrics saved to c:\Developer\Congreso_NLP\crispacion-congreso\data\processed\polarization_metrics.csv

Dataset shape: (144209, 8)

Metrics summary:
                year  polarization_lexicon_freq  accusations_per_sentence_pct
count  144209.000000              144209.000000                      144209.0
mean     2001.640674                 182.434328                           0.0
std        12.585061                 130.551971                           0.0
min      1979.000000                   5.000000                           0.0
25%      1991.000000                  92.000000                           0.0
50%      2002.000000                 140.000000                           0.0
75%      2012.000000                 246.000000                           0.0
max      2024.000000                 660.000000                           0.0
